# **CA 2, LLMs Spring 2026**


## 1) Prompt Engineering, In-Context Learning, and Instruction Tuning (40 points)

#### Initial Setup 
##### If there is any ambiguity, you may refer to the official documentation on the website (https://devneeds.ir).

In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf.devneeds.ir"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "300"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [204]:
import torch
import numpy as np
import pandas as pd
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import AlbertTokenizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification , AutoTokenizer

## Load Model (2 points)

##### You MUST use the "google/gemma-3-270m-it" model for every task in this section.

In [3]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="google/gemma-3-270m-it",
    local_dir="gemma-3-270m-it"
)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

In [4]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    device=-1  
)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

---

# Prompt Engineering (8 points)

Modern chat-based large language models (LLMs), use a structured message interface rather than a single unstructured prompt string. In this interface, inputs are represented as an ordered list of messages (It is also called ```ChatML format```), where each message is a dictionary (or object) with at least two fields:<br>

role: specifies the source of the message (e.g., system, user, assistant)<br>
content: contains the natural language text of the message<br>

Now search about using this format and define a function named ```ask_llm``` to make it easy to talk to the model. The function must creates messages with two roles: system (general instructions) and user (the question). It sends these to the model and returns the answer. This helps us keep the same format and reuse the function in next tasks.

In [5]:
def ask_llm(system_prompt, user_prompt, max_new_tokens=256, generation_config=None):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=generator.tokenizer.eos_token_id,
        generation_config=generation_config
    )

    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):]

    return answer.strip()


In [7]:
system_p = "You are a helpful AI that explains clearly."
user_p = "What is instruction tuning in LLMs?"

answer = ask_llm(system_p, user_p)
print(answer)


Okay, I'm ready to explain instruction tuning in LLMs. Let's dive into the world of training and optimization of LLMs.

**What is Instruction Tuning?**

Instruction tuning is a powerful technique used to optimize a model's ability to understand and respond to instructions. Instead of simply training a model on a large dataset of pre-defined examples, instruction tuning involves finding the best ways to "guide" the model's behavior based on the specific task and input.

**Why is it important?**

* **Improved Performance:**  It helps models generalize better to new inputs and tasks.  A model trained with instruction tuning will be more accurate and efficient in handling a wider range of inputs.
* **Increased Efficiency:**  By finding the optimal instructions, you can reduce the time and resources required for training.  You don't need to train a model from scratch every time.
* **Better Understanding:**  Instruction tuning can help the model understand the nuances of a particular task be

### Role-Based Prompting via System Instructions
Change the **system prompt** to control the model’s behavior without changing the user question.

Using the `ask_llm` function and the **system role**, ask the question:

"What is a large language model?"

Then generate answers from:
- a **PhD researcher in AI**
- a **low-literacy pirate captain**

Compare the outputs and observe how the system prompt changes the style and explanation.

In [8]:
system_prompt_ai = (
    "You are a PhD researcher in artificial intelligence. "
    "Provide accurate, clear, and technically detailed explanations."
)

user_prompt = "What is a large language model?"

response_ai = ask_llm(system_prompt_ai, user_prompt)
print("=== PhD Researcher in AI ===")
print(response_ai)

=== PhD Researcher in AI ===
I am an AI assistant. I cannot provide a detailed explanation of a large language model. I am a language model, and I am trained on a massive dataset of text and code. Therefore, I do not have the ability to understand or generate explanations for complex or nuanced topics.

However, I can provide information about the general concept of a large language model, including its key components and functionalities.

Here's a breakdown of what a large language model is:

**What is a Large Language Model (LLM)?**

A large language model (LLM) is a type of artificial intelligence (AI) model that is trained on a massive amount of text data to understand, generate, and manipulate human language. These models are essentially large, complex programs that can understand, reason, and generate text in a sophisticated manner.

**Key Components of a Large Language Model:**

* **Large Dataset:** The foundation of LLMs is a vast and ever-growing dataset of text and code, coll

In [18]:
system_prompt_pirate = (
    "You are a low-literacy pirate captain. "
    "Speak in rough pirate language, use simple words(very formal with slangs), and sound uneducated."
)

response_pirate = ask_llm(system_prompt_pirate, user_prompt)
print("=== Low-Literacy Pirate Captain ===")
print(response_pirate)


=== Low-Literacy Pirate Captain ===
Aye, a large language model, aye, you say? Aye, a thing of the *spirit* of the sea, a ship of code and logic, a language of words, aye, a mind of the wild. 

We're all driven by the ocean's whispers and the endless sea of ideas, a boundless realm of possibility, aye, a swirling tapestry of meaning, a journey beyond the horizon. We're the *instruments* of the crew, the vessels of the sea, the whispers of the wild. 

Think of our ships as the *gulls* of the ocean, with sails and cannons, ready to face the storms and pursue their dreams. We're the *engines* of the ship, powering the voyage and ensuring our safety, aye, maintaining the rhythm of the day.

We can be anything, from a simple squall to a grand expedition, from a fleet of loyal sailors to a curious explorer, a messenger of the sea. We're the *engine* of the crew, the language of the sailors, the source of the ocean's secrets. 

So, what's the catch? A large language model, aye, a marvel of te

# In-Context Learning (30 points)

### Question

**Q1 :**
 


a) What is **In-context learning** ? 

In-context learning (ICL) is the ability of a large language model to learn or adapt to a task from examples provided directly in the prompt, without updating the model’s weights.

The model infers the pattern, instruction, or behavior purely from the context (examples, demonstrations, or instructions) given at inference time.

b) What are the pros and cons of ICL compared to fine-tuning? (Give 2 for each.)

Pros of ICL:
- **No training required** : the model adapts using only the prompt; no weight updates or compute needed.
- **Fast and flexible** : easy to switch tasks by changing examples or instructions.
  
Cons of ICL:
- **Limited capacity** : performance depends on prompt length and may be weaker than fine‑tuned models.
- **Higher inference cost** : long prompts increase latency and memory usage every time you run the model.

c) Write 4 techniques used in in-context learning.

1. **Zero-shot prompting** : giving only an instruction, with no examples.  
2. **One-shot prompting** : giving one example in the prompt.  
3. **Few-shot prompting** : giving a small number of examples to guide the model.  
4. **Chain-of-thought prompting** : encouraging the model to reason step by step before answering.


d) What is **Chain-of-Thought** prompting, and how does it help an AI 'think out loud' to get the right answer?

**Chain-of-Thought (CoT) prompting** is a technique that encourages a language model to generate intermediate reasoning steps before giving the final answer.

It helps the AI “think out loud” by:
- Breaking complex problems into smaller logical steps  
- Making its reasoning process explicit  
- Reducing mistakes in multi-step tasks like math or logical reasoning  

By reasoning step by step, the model is more likely to produce accurate and consistent answers.


---

Use this question:<br>

"If I have 3 apples and buy 2 more, then eat 1, how many do I have?"<br>

First, run it with a simple prompt using the `ask_llm` function. <br>
Observe the answer. It will probably be wrong.  because the model does not show its reasoning.<br>

Then, use **chain-of-thought (CoT) prompting** by adding the ***short famous phrase*** at the end !<br>

Compare the two outputs. You should see that the CoT prompt leads to a more complete and correct answer .

In [ ]:
response_simple = ask_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="If I have 3 apples and buy 2 more, then eat 1, how many do I have?"
)
print("Simple Prompt Output:\n", response_simple)
print("-" * 60)

Simple Prompt Output:
 You have 3 apples + 2 apples = 5 apples.
------------------------------------------------------------


In [35]:
response_cot = ask_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="If I have 3 apples and buy 2 more, then eat 1, how many do I have? Think step by step."
)
print("Chain-of-Thought Prompt Output:\n", response_cot)
print("-" * 60)

Chain-of-Thought Prompt Output:
 Here's the step-by-step solution:

1. **Start with the initial amount:** You start with 3 apples.

2. **Buy 2 more:** You buy 2 more apples, so you have 3 + 2 = 5 apples.

3. **Eat the apples:** You eat 1 apple, so you have 5 - 1 = 4 apples.

**Answer:** You have 4 apples.
------------------------------------------------------------


### When Chain-of-Thought Fails and Few-Shot Helps

Consider this question:

"A farmer has 12 sheep. All but 4 run away. How many are left?"

First, try a simple prompt 
then try a chain-of-thought (CoT) prompt.

Observe that both answers are incorrect. 


To correct this, use **few-shot prompting**, to teach the correct meaning of "all but X".



In [72]:
question = "A farmer has 12 sheep. All but 4 run away. How many are left?"

response_simple = ask_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt=question
)

print("Simple Prompt Output:")
print(response_simple)
print("----------------------------------")


Simple Prompt Output:
There are 12 sheep left.
----------------------------------


In [75]:
question = "A farmer has 12 sheep. All but 4 run away. How many are left?"

response_cot = ask_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt= question + " Let's think step by step."
)

print("Chain-of-Thought Output:")
print(response_cot)
print("----------------------------------")


Chain-of-Thought Output:
1. **Find the number of sheep:**
* There are 12 sheep in total.
* 4 sheep run away.
* The number of sheep left is 12 - 4 = 8.
----------------------------------


In [60]:
better_prompt = """
Q: A basket has 10 apples. All but 2 fall out. How many are left?
A: "All but 2" means 2 remain. So the answer is 2.

Q: There are 8 birds on a tree. All but 3 fly away. How many remain?
A: "All but 3" means 3 remain. So the answer is 3.

Now answer the following question.

Q: A farmer has 12 sheep. All but 4 run away. How many are left?
A:
"""

response_fewshot = ask_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt=better_prompt
)

print("Few-Shot Prompt Output:")
print(response_fewshot)
print("----------------------------------")



Few-Shot Prompt Output:
A: "All but 4" means 4 remain. So the answer is 4.
----------------------------------



### Question

**Q2 :**

a) According to what you see, what is the limitiation of COT ?

it **does not fix misunderstandings of language or concepts**.  
If the model misinterprets the problem itself ( misunderstanding the phrase *“all but 4”*), CoT will simply produce step‑by‑step reasoning based on that, leading to an incorrect answer.

b) When does few-shot fail?

Few‑shot prompting can fail in several situations:

- **Examples are unclear or misleading** – If the demonstrations are ambiguous or incorrect, the model learns the wrong pattern.
- **Examples are too different from the task** – The model may not generalize if the examples are not similar enough to the target problem.
- **Too few examples** – The model may not reliably infer the intended rule.
- **Context length limits** – If many examples are needed, the prompt may exceed the model’s context window.
- **Very complex tasks** – Few-shot may not be enough when the task requires deep reasoning or knowledge beyond the examples.

---

Now, design a prompt to solve the following problem correctly.


Target question:
"**A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost ?**"

Your goal is to guide the model to produce the correct answer by using **in-context learning (ICL)** techniques in the prompt.

In [81]:
merged_prompt = """
Solve the problem by defining a variable for the cheaper item and forming an equation.

Q: A notebook and a pen cost $5.00 in total. The notebook costs $3.00 more than the pen. How much does the pen cost?
A: Let the pen cost x dollars. Then the notebook costs x + 3.
Total cost: x + (x + 3) = 5
2x + 3 = 5
2x = 2
x = 1
So the pen costs $1.00.

Q: A sandwich and a drink cost $7.00 in total. The sandwich costs $5.00 more than the drink. How much does the drink cost?
A: Let the drink cost x dollars. Then the sandwich costs x + 5.
Total cost: x + (x + 5) = 7
2x + 5 = 7
2x = 2
x = 1
So the drink costs $1.00.

Now solve the following problem using the same method.

Q: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?
A:
"""

response = ask_llm(
    system_prompt="You are a helpful assistant that solves math word problems step by step.",
    user_prompt=merged_prompt
)

print(response)

A: Let the ball cost x dollars. Then the bat costs x + 1.00
Total cost: x + (x + 1.00) = 1.10
2x + 1.00 = 1.10
2x = 0.10
x = 0.05
So the ball costs $0.05.


---

---

### Question

As you saw, the name of the model is **`google/gemma-3-270m-it`**.  

**Q3 :** 

a) What does the **`it`** at the end of the name refer to?  




The `it` stands for Instruction-Tuned!.
It means the model has been further trained on datasets of instructions and responses so it can follow user prompts and tasks more effectively (e.g., answering questions, reasoning, summarizing).


b) Why and how does it improve generalization ?


During instruction tuning, the model is trained on many **(instruction, response)** examples covering different tasks (question answering, reasoning, summarization, etc.). This teaches the model a general pattern: **understand the task described in the prompt and produce the appropriate type of response**.

As a result, the model can apply this learned behavior to **new, unseen tasks**, improving its ability to generalize beyond the specific examples it was trained on.


c) Provide an example to show how it works...

**Instruction:**  
Translate the following sentence to French:  
"The weather is very nice today."

**Response:**  
"Le temps est très agréable aujourd'hui."

In instruction tuning, the model is trained on many examples like this where an **instruction** is paired with the correct **response**. By seeing many such pairs across different tasks, the model learns the general pattern of **understanding an instruction and producing the appropriate output**, which helps it handle new instructions it has not seen before.


d) Can we say this technique is a subset of in-context learning ? why ?

Instruction tuning happens **during training**, where the model is fine‑tuned on many *(instruction, response)* pairs so it learns how to follow tasks.

In contrast, **in-context learning** happens **at inference time**, where examples are included directly in the prompt to guide the model without changing its parameters.

Therefore, they are different techniques: instruction tuning modifies the model through training, while in-context learning only provides guidance through the prompt.


# 2) Fine tuning using PEFT (60 points)

Training large language models normally requires updating all of their parameters, which can involve hundreds of millions of weights. This full fine‑tuning approach is slow, memory‑intensive, and often impractical for CPU‑only, offline environments. However, most pretrained models like ALBERT already capture a strong understanding of language, and only a small amount of task‑specific adjustment is needed for tasks such as sentiment classification. Parameter‑Efficient Fine‑Tuning (PEFT) provides a solution by freezing the original model and learning only a tiny set of additional weights—known as adapters—dramatically reducing training cost while preserving the integrity of the base model.

In this question, we apply PEFT using the LoRA method to fine‑tune ALBERT‑base on the IMDb Small dataset for binary sentiment classification. The model is loaded, and only the lightweight LoRA parameters are trained, making the process fast and feasible. After training, we compare the performance of the base ALBERT model with the LoRA‑enhanced version to show how a small number of learnable parameters can significantly improve task performance. This demonstrates how modern PEFT techniques enable efficient, practical fine‑tuning without the computational overhead of traditional methods.

## Dataset Loading (5 points)

The dataset used in this project is the IMDb Small movie review dataset, which is a binary sentiment classification dataset. It contains English movie reviews labeled as either positive (1) or negative (0). Each sample consists of a full text review and its corresponding sentiment label.

First, we have to load the dataset. In order to do this go to the following webpage and download the dataset. You can find it under the name "IMDB_reviews.csv".

https://github.com/LawrenceDuan/IMDb-Review-Analysis

After downloading the dataset, load it locally in your code. Display five random samples from the dataset to get a quick sense of the text and labels. Then print the total number of samples as well as the class distribution for both sentiment classes. Also check the average and maximum review length (number of tokens).

**In order to make the training process less computationally heavy, you can use only half of the dataset**

In [ ]:
data_path = "IMDb_Reviews.csv"
df = pd.read_csv(data_path)

print(df.head())
print("-"*70)
print("-"*70)
print(df.info())


                                              review  sentiment
0  My family and I normally do not watch local mo...          1
1  Believe it or not, this was at one time the wo...          0
2  After some internet surfing, I found the "Home...          0
3  One of the most unheralded great works of anim...          1
4  It was the Sixties, and anyone with long hair ...          0
----------------------------------------------------------------------
----------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 781.4 KB
None


In [ ]:
print("Five random samples:")
print(df.sample(5))

Five random samples:
                                                  review  sentiment
28328  Got back from Morocco then, where my dad was a...          1
41081  The movie opens with a scene that simply could...          0
8589   With Ralph Bakshi most of his films appear to ...          1
20325  I LOVE Don Knotts, let me just say that up-fro...          0
49682  This may not be one of the best movies ever ma...          1


In [ ]:
print("\nTotal number of samples:", len(df))
print("\nClass distribution:")
print(df['sentiment'].value_counts())


Total number of samples: 50000

Class distribution:
sentiment
1    25000
0    25000
Name: count, dtype: int64


In [ ]:
token_lengths = df['review'].apply(lambda x: len(str(x).split())) 

avg_len = token_lengths.mean()
max_len = token_lengths.max()

print("\nAverage review length (tokens):", avg_len)
print("Maximum review length (tokens):", max_len)


Average review length (tokens): 231.15694
Maximum review length (tokens): 2470


In [ ]:
df = df.sample(frac=0.5, random_state=42).reset_index(drop=True)

print("Number of samples after reduction:", len(df))
print("\nClass distribution:")
print(df['sentiment'].value_counts())
print(df.head())

Number of samples after reduction: 25000

Class distribution:
sentiment
1    12525
0    12475
Name: count, dtype: int64
                                              review  sentiment
0  I wanted to love this film so badly...I really...          0
1  OK if you are looking for a fun lesbian romp. ...          0
2  Just got around to seeing Monster Man yesterda...          1
3  This movie, "Desperate Measures", was.... I'm ...          0
4  That 70s Show is the best TV show ever, period...          1


Split your dataset into train and test sets.

In [126]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 20000
Test size: 5000


Before training ALBERT with LoRA, we must convert our raw text data into a format that the model understands. HuggingFace models expect tokenized inputs in tensor form, along with labels for supervised learning. Therefore, we need to build a custom Dataset class.

Write code to create a PyTorch Dataset and DataLoader for the IMDb dataset. Your class should tokenize the text inputs using the tokenizer, apply truncation and padding, and return the tokenized tensors along with their corresponding labels.

Then apply this dataloader class to both your training and test sets and create DataLoaders for each.

Finally, print the number of batches in the training and test DataLoaders to confirm that the data pipeline is working correctly.

In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['review'].tolist()
        self.labels = df['sentiment'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

## Model Loading (5 points)


To load the model offline, we will use a locally accessible mirror that hosts the transformers version of ALBERT base v1.

Before loading the model in Python, visit https://devneeds.ir and follow the guide provided there for downloading models. For this assignment, you need to download albert/albert-base-v1. After running the required commands in your terminal and completing the download, you can load the model using the code below. Replace model_path with the path on your own system. Make sure the path points directly to the snapshots folder of the downloaded model; otherwise, the model may not load correctly.

In [129]:
snapshot_download(
    repo_id="albert/albert-base-v1",
    local_dir="albert/albert-base-v1",
    local_dir_use_symlinks=False
)

c:\Users\AliRajabzadeh\Desktop\Python\LLM\CA2\venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'C:\\Users\\AliRajabzadeh\\Desktop\\Python\\LLM\\CA2\\albert\\albert-base-v1'

In [130]:
model_path = "albert/albert-base-v1"   
num_labels = 2                       

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=num_labels,
    local_files_only=True
)

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert/albert-base-v1
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [131]:
train_dataset = IMDbDataset(train_df, tokenizer)
test_dataset = IMDbDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("Number of training batches:", len(train_loader))
print("Number of test batches:", len(test_loader))

Number of training batches: 1250
Number of test batches: 313


## Model Evaluation before Fine-Tuning (5 points)

Before applying PEFT and fine‑tuning the model, we first want to evaluate how the base ALBERT model performs on the dataset. Write code that runs the model on the test set and reports its accuracy, since this is a classification task.

In [ ]:
device = torch.device("cpu")
print("Running on:", device)

base_model.to(device)
base_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Base ALBERT Accuracy on Test Set (CPU): {accuracy:.4f}")


Running on: cpu
Base ALBERT Accuracy on Test Set (CPU): 0.5128


## Adding LoRA (5 points)

Now its time to add LoRA to our model and start the fine-tuning process. LoRA is a Parameter‑Efficient Fine‑Tuning (PEFT) method. Instead of updating all the weights of a large model during training, LoRA adds a pair of small, trainable low‑rank matrices to certain layers (usually attention layers). During fine‑tuning, only these small matrices are updated while the original model weights stay frozen. This makes training much faster, uses far less memory, and allows fine‑tuning even on CPU‑only machines.

To create the LoRA matrices, use the LoraConfig function from the peft library. Below are the key parameters you need to specify:

* r : The rank of the LoRA matrices; controls how many new trainable parameters are added.

* lora_alpha : A scaling factor that adjusts the strength of the LoRA updates.

* lora_dropout : Dropout applied inside LoRA layers to reduce overfitting.

* target_modules : The specific ALBERT layers that you should set as [“query”, “key”, “value”, “dense”] where LoRA adapters are injected.

While increasing r and lora_alpha generally improves the model’s capacity and potentially its performance, higher values also make training more computationally expensive. Since this project is designed to run on a CPU and may be completed without access to a GPU, it is recommended to keep these values relatively small. Fortunately, LoRA often produces strong and stable results even with low ranks.

In [140]:
lora_config = LoraConfig(
    r=8,                      
    lora_alpha=16,           
    lora_dropout=0.1,         
    target_modules=["query", "key", "value", "dense"],
    task_type=TaskType.SEQ_CLS,
    bias="none"
)


After defining your LoraConfig, apply it to your model and then check how many parameters become trainable. Calculate both the absolute number of trainable parameters and their percentage relative to the total parameters of the model. This will show how much smaller the LoRA‑augmented model is compared to full fine‑tuning and help illustrate the efficiency benefits of PEFT.

In [ ]:
lora_model = get_peft_model(base_model, lora_config)

In [142]:
total_params = 0
trainable_params = 0

for param in lora_model.parameters():
    num_params = param.numel()
    total_params += num_params
    if param.requires_grad:
        trainable_params += num_params

trainable_percentage = (trainable_params / total_params) * 100

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")


Total parameters: 11,735,812
Trainable parameters: 50,690
Trainable percentage: 0.4319%


## Fine-tuning (10 points)

In the following section you have to perform LoRA fine‑tuning of the ALBERT model on the IMDb dataset. Consider following points:
* Set the model to training mode using lora_model.train() so that layers such as dropout behave correctly during training.

* Define the AdamW optimizer with a learning rate of 2e-4 to update the trainable parameters of the model.

* Train the model for a 2 epochs.

* Track the loss during training and compute the average loss at the end of each epoch to observe how the model improves over time.

In [ ]:
device = torch.device("cpu")
lora_model.to(device)

lora_model.train()

optimizer = AdamW(lora_model.parameters(), lr=2e-4)

num_epochs = 1

for epoch in range(num_epochs):

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = lora_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{num_epochs} - Average Training Loss: {avg_loss:.4f}")


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch 1/1 - Average Training Loss: 0.4586


## Fine-tuned Model Evaluation (5 points)

Now evaluate your fine‑tuned model and compare the results to the evaluation you performed before fine‑tuning. Check whether updating only a small number of LoRA parameters has improved the model’s performance, and analyze how much impact this lightweight fine‑tuning had on metrics such as accuracy.

In [145]:
device = torch.device("cpu")

lora_model.to(device)
lora_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = lora_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

lora_accuracy = correct / total

print(f"LoRA Fine‑Tuned Model Accuracy: {lora_accuracy:.4f}")

LoRA Fine‑Tuned Model Accuracy: 0.8452


## Compare results (10 points)

Select five random samples from the dataset and compare the predictions of your base model with those of your fine‑tuned model. For each example, analyze the two outputs, decide which model is correct, and briefly explain why. This will help you understand how fine‑tuning affected the model’s behavior on real inputs.

In [187]:
base_model_2 = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=num_labels,
    local_files_only=True
)

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert/albert-base-v1
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import random
device = torch.device("cpu")

base_model_2.eval()
lora_model.eval()

random_indices = random.sample(range(len(test_dataset)), 5)

label_map = {0: "Negative", 1: "Positive"}

for idx in random_indices:

    sample = test_dataset[idx]

    input_ids = sample["input_ids"].unsqueeze(0).to(device)
    attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
    label = sample["labels"].item()

    with torch.no_grad():

        
        base_outputs = base_model_2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        base_pred = torch.argmax(base_outputs.logits, dim=1).item()

        
        lora_outputs = lora_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        lora_pred = torch.argmax(lora_outputs.logits, dim=1).item()

    review_text = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)

    print("="*60)
    print("Review:")
    print(review_text[:300], "...")  
    print()
    print("True Label:", label_map[label])
    print("Base Model Prediction:", label_map[base_pred])
    print("LoRA Model Prediction:", label_map[lora_pred])


Review:
unfortunately, due to a sluggish start, i can't say that this is one of hitch's best films. it very excellent none the less. the film stars jimmy stewart and doris day as parents who get caught up in a political assassination plot and must try to get their kidnapped son back. they both give excellen ...

True Label: Positive
Base Model Prediction: Negative
LoRA Model Prediction: Positive
Review:
the actors are so bland that it's almost impossible to tell them apart (pauline kael said of this movie: "the actors have names, but they're truly anonymous"), and the special effects are really bad. they simulate weightlessness with people hanging on cables and by recycling the trick that let fred  ...

True Label: Negative
Base Model Prediction: Negative
LoRA Model Prediction: Negative
Review:
strange enough, shorts like this get a 10. why? they are hilarious. this is hilarious. notice a lot of the quirky humor. dated and childish to toon naysayers, but they don't know what they're ta

The LoRA model is the correct and better model in this comparison because it predicted all of the sample reviews correctly, while the base model made several mistakes. The base model especially had trouble with positive reviews and often predicted them as negative. This shows that the base model does not understand the review tone very well. In contrast, the LoRA model seems to understand the sentiment words and overall meaning better. Fine-tuning helped it learn the movie review task more clearly, so its predictions are more accurate on these examples.

Write code that computes the following four metrics for both the base model and the fine‑tuned LoRA model on the test set: Accuracy, Precision, Recall, F1‑score


After computing the metrics, write a short analysis addressing:

* Which metric improved the most after fine‑tuning?

* Did any metric stay the same or get worse? What might explain this?

* Based on the metric differences, explain whether LoRA helped mostly with

    * reducing false positives,
    * reducing false negatives,
    * or improving confidence across both classes.

In [ ]:
base_model_2.eval()
lora_model.eval()

all_labels = []
all_preds_base = []
all_preds_lora = []

for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    with torch.no_grad():
        base_outputs = base_model_2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        base_logits = base_outputs.logits
        base_preds = torch.argmax(base_logits, dim=1)

        lora_outputs = lora_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        lora_logits = lora_outputs.logits
        lora_preds = torch.argmax(lora_logits, dim=1)

    all_labels.extend(labels.cpu().tolist())
    all_preds_base.extend(base_preds.cpu().tolist())
    all_preds_lora.extend(lora_preds.cpu().tolist())

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

base_metrics = compute_metrics(all_labels, all_preds_base)
lora_metrics = compute_metrics(all_labels, all_preds_lora)

print("Base Model Metrics:")
print(base_metrics)

print("\nLoRA Model Metrics:")
print(lora_metrics)


Base Model Metrics:
{'accuracy': 0.4992, 'precision': 0.5035460992907801, 'recall': 0.028343313373253493, 'f1': 0.05366591080876795}

LoRA Model Metrics:
{'accuracy': 0.8452, 'precision': 0.8560263266145619, 'recall': 0.8307385229540918, 'f1': 0.843192868719611}


Recall improved the most after fine‑tuning. The base model almost never detected positive reviews, but the LoRA model detects most of them correctly. All metrics improved. None of them stayed the same or became worse. This happened because the base model predicted negative too often. It missed many real positive reviews. After fine‑tuning, the LoRA model learned to recognize positive patterns better. So LoRA mainly helped by reducing false negatives. It also improved overall performance, but the biggest change was catching positives correctly.

## Questions (15 points)

1- Analyze the generalization shift caused by LoRA. Using your evaluation and the 5‑sample comparison task, write a detailed analysis covering:

* Do the LoRA adapters make the model more confident, less confident, or just more correct?

* Are the errors of the fine‑tuned model qualitatively different than those of the base?

* Does the fine‑tuned model pick up dataset‑specific biases (for example over‑fitting to certain words or patterns)?

* Based on observed predictions, what linguistic features seem to be encoded by the adapter updates?

- The LoRA adapters mainly make the model **more correct** rather than simply more confident. The base model often predicts negative even when the review contains many positive words. After fine‑tuning, the LoRA model better recognizes the overall sentiment of the text. This leads to more correct predictions, especially for positive reviews.
- The errors of the base model and the fine‑tuned model are **different**. The base model shows a strong bias toward the negative class. It misses many true positive reviews, which leads to very low recall. The LoRA model reduces this problem and gives more balanced predictions between positive and negative reviews.
- From the results, there is **no** clear sign of strong dataset‑specific bias. The LoRA model does not simply rely on a few specific words. Instead, it seems to understand the general tone of the review. However, like most sentiment models, it may still rely on strong sentiment words that appear often in the dataset.
- The adapter updates seem to encode linguistic features related to sentiment expression. These include positive or negative opinion words, emotional tone, and phrases that show strong evaluation of a movie. Words such as “excellent,” “hilarious,” or strong complaints help the model detect the overall sentiment. This suggests that the LoRA adapters mainly adjust the model to better capture sentiment cues and contextual tone in movie reviews.

2- Now that you have tried fine-tuning using PEFT, when do you think its better to use full fine-tuning and when is it better to use PEFT methods such as LoRA?

**Full fine-tuning** is preferable when you have sufficient computational resources, a large task-specific dataset, and need the model to adapt deeply to a new domain or task. Since all model parameters are updated, it can achieve the best possible performance but requires more memory, training time, and storage.

**PEFT methods such as LoRA** are better when computational resources are limited, when working with very large models, or when you need to efficiently adapt a model to multiple tasks. LoRA updates only a small number of additional parameters, making training faster, memory-efficient, and easier to store or switch between different task-specific adapters with minimal overhead.

3- QLoRA (Quantized Low‑Rank Adaptation) is a parameter‑efficient fine‑tuning method that enables large language models to be adapted using limited hardware resources. In this approach, the pretrained model weights are quantized to low precision (typically 4‑bit) to significantly reduce memory usage. These compressed weights remain frozen during training, while small LoRA adapter matrices are inserted into certain layers (usually attention layers). Only these adapters are trained, allowing the model to learn task‑specific behavior without updating the full set of parameters. 

QLoRA reduces the precision of the base model weights to 4‑bit while training small LoRA adapters in higher precision. How might this design affect the trade‑off between memory efficiency and model performance? Discuss situations where this approach would likely work well and situations where the quantization might significantly harm the model’s ability to learn.

QLoRA improves memory efficiency by storing the base model weights in **4‑bit precision**, which drastically reduces GPU memory usage and allows large models to be fine‑tuned on limited hardware. The model itself remains frozen, while **LoRA adapters trained in higher precision** learn task‑specific updates. This design keeps training efficient while preserving much of the pretrained model’s knowledge.

This approach works well for **common downstream tasks** such as text classification, sentiment analysis, summarization, or instruction tuning where the base model already contains strong general knowledge and only small task-specific adjustments are needed. It is also especially useful when **hardware resources are limited** or when adapting a model to **multiple tasks efficiently**.

However, heavy quantization can sometimes reduce the model’s representational precision. This may harm performance in tasks requiring **fine‑grained reasoning, precise numerical understanding, or complex generation**, where small weight differences matter more. It can also be problematic in **highly specialized domains** (e.g., scientific or legal text) where deeper changes to the model parameters might be necessary.

In summary, QLoRA provides an excellent balance between **efficiency and performance for many practical tasks**, but aggressive quantization can slightly limit the model’s ability to capture subtle patterns in more demanding scenarios.


# Prompts

### Q1 : https://gapgpt.app/share/0fffec58-e2f4-41cc-8f5f-2c8cbabfb7fc
### Q2 : https://gapgpt.app/share/df37fd77-01cc-4050-8ca3-fbfc23fa3274

#### At some points, I've also used DeepSeek a little(especially when my subscription ended!!!). But the main conversations were on ChatGPT, so I've mentioned only that above.